In [1]:
import sys

print("Python version:", sys.version)
print("Python location:", sys.executable)

Python version: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
Python location: C:\Users\ASUS\anaconda3\python.exe


In [2]:
%pip install fastmcp

   ---------------------------------------- 0.0/931.0 kB ? eta -:--:--
   ---------------------------------------- 931.0/931.0 kB 15.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 12.4 MB/s  0:00:00

   ---- -----------------------------------  3/29 [opentelemetry-api]
   ----- ----------------------------------  4/29 [jsonref]
   ------ ---------------------------------  5/29 [httpcore2]
   -------- -------------------------------  6/29 [griffelib]
   -------- -------------------------------  6/29 [griffelib]
   --------- ------------------------------  7/29 [exceptiongroup]
   ------------ ---------------------------  9/29 [dnspython]
   ------------ ---------------------------  9/29 [dnspython]
   ------------ ---------------------------  9/29 [dnspython]
   ------------ ---------------------------  9/29 [dnspython]
   ------------ ---------------------------  9/29 [dnspython]
   -----------

In [3]:
from importlib.metadata import version
import fastmcp

print("FastMCP installed successfully!")
print("Version:", version("fastmcp"))

FastMCP installed successfully!
Version: 4.1.3


In [4]:

from fastmcp import FastMCP

# Create our MCP server
mcp = FastMCP("LeadPilot")


def qualify_lead_logic(
    customer_name: str,
    budget: float,
    interest_level: str,
    urgency: str
) -> dict:
    """Calculate a lead score and recommend its priority."""

    # Validate customer inputs
    if not customer_name.strip():
        raise ValueError("Customer name cannot be empty.")

    if budget < 0:
        raise ValueError("Budget cannot be negative.")

    interest_level = interest_level.lower().strip()
    urgency = urgency.lower().strip()

    if interest_level not in ["hot", "warm", "cold"]:
        raise ValueError("Interest must be hot, warm, or cold.")

    if urgency not in ["high", "medium", "low"]:
        raise ValueError("Urgency must be high, medium, or low.")

    # Calculate points based on customer details
    score = 0

    if budget >= 50000:
        score += 30
    elif budget >= 20000:
        score += 20
    else:
        score += 10

    interest_points = {"hot": 40, "warm": 25, "cold": 10}
    urgency_points = {"high": 30, "medium": 20, "low": 10}

    score += interest_points[interest_level]
    score += urgency_points[urgency]

    # Assign lead priority
    if score >= 80:
        priority = "High"
        next_action = "Contact the customer as soon as possible."
    elif score >= 60:
        priority = "Medium"
        next_action = "Follow up with the customer soon."
    else:
        priority = "Low"
        next_action = "Send more information and nurture the lead."

    return {
        "customer_name": customer_name,
        "lead_score": score,
        "priority": priority,
        "recommended_action": next_action
    }


# Expose the function as an MCP tool
@mcp.tool
def qualify_lead(
    customer_name: str,
    budget: float,
    interest_level: str,
    urgency: str
) -> dict:
    """Qualify a customer lead and recommend a follow-up action."""
    return qualify_lead_logic(
        customer_name, budget, interest_level, urgency
    )


print("LeadPilot MCP tool registered!")
print("Server name:", mcp.name)


LeadPilot MCP tool registered!
Server name: LeadPilot


In [5]:
result = qualify_lead_logic(
    customer_name="Priya",
    budget=60000,
    interest_level="hot",
    urgency="high"
)

print(result)

{'customer_name': 'Priya', 'lead_score': 100, 'priority': 'High', 'recommended_action': 'Contact the customer as soon as possible.'}


In [6]:

from fastmcp import Client
import asyncio

async def test_leadpilot():
    async with Client(mcp) as client:
        # Check which tools the server exposes
        tools = await client.list_tools()
        print("Available MCP tools:", [tool.name for tool in tools])

        # Call the tool through the MCP client
        result = await client.call_tool(
            "qualify_lead",
            {
                "customer_name": "Priya",
                "budget": 60000,
                "interest_level": "hot",
                "urgency": "high"
            }
        )

        print("\nMCP tool call result:")
        print(result.data)


await test_leadpilot()


Available MCP tools: ['qualify_lead']

MCP tool call result:
{'customer_name': 'Priya', 'lead_score': 100, 'priority': 'High', 'recommended_action': 'Contact the customer as soon as possible.'}


In [7]:

test_cases = [
    {
        "customer_name": "Aisha",
        "budget": 60000,
        "interest_level": "hot",
        "urgency": "high"
    },
    {
        "customer_name": "Rahul",
        "budget": 30000,
        "interest_level": "warm",
        "urgency": "medium"
    },
    {
        "customer_name": "John",
        "budget": 5000,
        "interest_level": "cold",
        "urgency": "low"
    }
]

for customer in test_cases:
    result = qualify_lead_logic(**customer)
    print(
        customer["customer_name"],
        "-> Score:", result["lead_score"],
        "| Priority:", result["priority"]
    )


Aisha -> Score: 100 | Priority: High
Rahul -> Score: 65 | Priority: Medium
John -> Score: 30 | Priority: Low


In [8]:

invalid_tests = [
    {
        "customer_name": "Test User",
        "budget": -100,
        "interest_level": "hot",
        "urgency": "high"
    },
    {
        "customer_name": "Test User",
        "budget": 20000,
        "interest_level": "unknown",
        "urgency": "high"
    },
    {
        "customer_name": "Test User",
        "budget": 20000,
        "interest_level": "warm",
        "urgency": "urgent"
    }
]

for customer in invalid_tests:
    try:
        qualify_lead_logic(**customer)
        print("FAIL: Invalid input was accepted")
    except ValueError as error:
        print("PASS:", error)


PASS: Budget cannot be negative.
PASS: Interest must be hot, warm, or cold.
PASS: Urgency must be high, medium, or low.


In [9]:
%pip install fastapi

Note: you may need to restart the kernel to use updated packages.


In [10]:

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(
    title="LeadPilot API",
    description="API for lead qualification and prioritization",
    version="1.0.0"
)


class LeadRequest(BaseModel):
    customer_name: str = Field(min_length=1)
    budget: float = Field(ge=0)
    interest_level: str
    urgency: str


@app.post("/qualify-lead")
def qualify_lead_api(lead: LeadRequest):
    try:
        return qualify_lead_logic(**lead.model_dump())
    except ValueError as error:
        raise HTTPException(
            status_code=422,
            detail=str(error)
        )


print("FastAPI application created:", app.title)
print("Endpoint: POST /qualify-lead")


FastAPI application created: LeadPilot API
Endpoint: POST /qualify-lead


In [11]:

from fastapi.testclient import TestClient

client = TestClient(app)

response = client.post(
    "/qualify-lead",
    json={
        "customer_name": "Aisha",
        "budget": 60000,
        "interest_level": "hot",
        "urgency": "high"
    }
)

print("Status code:", response.status_code)
print("API response:", response.json())

assert response.status_code == 200
assert response.json()["lead_score"] == 100
assert response.json()["priority"] == "High"

print("\nAPI TEST PASSED!")


Status code: 200
API response: {'customer_name': 'Aisha', 'lead_score': 100, 'priority': 'High', 'recommended_action': 'Contact the customer as soon as possible.'}

API TEST PASSED!


In [12]:

invalid_api_tests = [
    {
        "customer_name": "Rahul",
        "budget": -500,
        "interest_level": "hot",
        "urgency": "high"
    },
    {
        "customer_name": "Aisha",
        "budget": 20000,
        "interest_level": "unknown",
        "urgency": "high"
    },
    {
        "customer_name": "John",
        "budget": 10000,
        "interest_level": "cold",
        "urgency": "urgent"
    }
]

for data in invalid_api_tests:
    response = client.post("/qualify-lead", json=data)

    if response.status_code == 422:
        print("PASS: Invalid request rejected")
    else:
        print("FAIL:", response.status_code, response.json())

print("\nInvalid API tests completed.")


PASS: Invalid request rejected
PASS: Invalid request rejected
PASS: Invalid request rejected

Invalid API tests completed.


In [14]:
from pathlib import Path
import sys

project_dir = Path.cwd() / "LeadPilot"
project_dir.mkdir(exist_ok=True)

print("Folder location:", project_dir)
print("Folder exists:", project_dir.exists())

Folder location: C:\Users\ASUS\Documents\LeadPilot
Folder exists: True


In [16]:

logic_code = '''
def qualify_lead_logic(
    customer_name: str,
    budget: float,
    interest_level: str,
    urgency: str
) -> dict:
    """Score a sales lead and recommend a follow-up action."""

    if not customer_name.strip():
        raise ValueError("Customer name cannot be empty.")

    if budget < 0:
        raise ValueError("Budget cannot be negative.")

    interest_level = interest_level.lower().strip()
    urgency = urgency.lower().strip()

    if interest_level not in ["hot", "warm", "cold"]:
        raise ValueError("Interest must be hot, warm, or cold.")

    if urgency not in ["high", "medium", "low"]:
        raise ValueError("Urgency must be high, medium, or low.")

    score = 0

    if budget >= 50000:
        score += 30
    elif budget >= 20000:
        score += 20
    else:
        score += 10

    score += {"hot": 40, "warm": 25, "cold": 10}[interest_level]
    score += {"high": 30, "medium": 20, "low": 10}[urgency]

    if score >= 80:
        priority = "High"
        action = "Contact the customer as soon as possible."
    elif score >= 60:
        priority = "Medium"
        action = "Follow up with the customer soon."
    else:
        priority = "Low"
        action = "Send more information and nurture the lead."

    return {
        "customer_name": customer_name,
        "lead_score": score,
        "priority": priority,
        "recommended_action": action
    }
'''

file_path = project_dir / "lead_logic.py"
file_path.write_text(logic_code.strip() + "\n", encoding="utf-8")

print("File created:", file_path)
print("File exists:", file_path.exists())


File created: C:\Users\ASUS\Documents\LeadPilot\lead_logic.py
File exists: True


In [17]:
import sys

sys.path.insert(0, str(project_dir))

from lead_logic import qualify_lead_logic

result = qualify_lead_logic(
    "Aisha", 60000, "hot", "high"
)

print(result)

{'customer_name': 'Aisha', 'lead_score': 100, 'priority': 'High', 'recommended_action': 'Contact the customer as soon as possible.'}


In [18]:

mcp_code = '''
from fastmcp import FastMCP
from lead_logic import qualify_lead_logic

mcp = FastMCP("LeadPilot")


@mcp.tool
def qualify_lead(
    customer_name: str,
    budget: float,
    interest_level: str,
    urgency: str
) -> dict:
    """Qualify a sales lead and recommend a follow-up action."""
    return qualify_lead_logic(
        customer_name=customer_name,
        budget=budget,
        interest_level=interest_level,
        urgency=urgency
    )
'''

mcp_file = project_dir / "mcp_server.py"
mcp_file.write_text(mcp_code.strip() + "\n", encoding="utf-8")

print("File created:", mcp_file)
print("File exists:", mcp_file.exists())


File created: C:\Users\ASUS\Documents\LeadPilot\mcp_server.py
File exists: True


In [19]:
import sys
sys.path.insert(0, str(project_dir))

from mcp_server import mcp

print("MCP server:", mcp.name)
print("MCP server imported successfully!")

MCP server: LeadPilot
MCP server imported successfully!


In [20]:

from fastmcp import Client

async def test_saved_mcp_server():
    async with Client(mcp) as client:
        tools = await client.list_tools()
        print("Available tools:", [tool.name for tool in tools])

        result = await client.call_tool(
            "qualify_lead",
            {
                "customer_name": "Aisha",
                "budget": 60000,
                "interest_level": "hot",
                "urgency": "high"
            }
        )

        print("Tool result:", result.data)


await test_saved_mcp_server()


Available tools: ['qualify_lead']
Tool result: {'customer_name': 'Aisha', 'lead_score': 100, 'priority': 'High', 'recommended_action': 'Contact the customer as soon as possible.'}


In [21]:

api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from lead_logic import qualify_lead_logic

app = FastAPI(
    title="LeadPilot API",
    description="API for lead qualification and prioritization",
    version="1.0.0"
)


class LeadRequest(BaseModel):
    customer_name: str = Field(min_length=1)
    budget: float = Field(ge=0)
    interest_level: str
    urgency: str


@app.post("/qualify-lead")
def qualify_lead_api(lead: LeadRequest):
    try:
        return qualify_lead_logic(**lead.model_dump())
    except ValueError as error:
        raise HTTPException(status_code=422, detail=str(error))
'''

api_file = project_dir / "api.py"
api_file.write_text(api_code.strip() + "\n", encoding="utf-8")

print("File created:", api_file)
print("File exists:", api_file.exists())


File created: C:\Users\ASUS\Documents\LeadPilot\api.py
File exists: True


In [22]:
from api import app
from fastapi.testclient import TestClient

api_client = TestClient(app)

response = api_client.post(
    "/qualify-lead",
    json={
        "customer_name": "Aisha",
        "budget": 60000,
        "interest_level": "hot",
        "urgency": "high"
    }
)

print("Status:", response.status_code)
print("Result:", response.json())

Status: 200
Result: {'customer_name': 'Aisha', 'lead_score': 100, 'priority': 'High', 'recommended_action': 'Contact the customer as soon as possible.'}


In [23]:
requirements = """fastmcp
fastapi
uvicorn
"""

requirements_file = project_dir / "requirements.txt"
requirements_file.write_text(requirements, encoding="utf-8")

print("File created:", requirements_file)
print(requirements_file.read_text())

File created: C:\Users\ASUS\Documents\LeadPilot\requirements.txt
fastmcp
fastapi
uvicorn



In [24]:

readme_content = """
# LeadPilot — AI-Powered Lead Qualification MCP Server

LeadPilot is a lead qualification prototype built with Python,
FastMCP, and FastAPI.

## Features

- Scores sales leads using budget, interest, and urgency.
- Classifies leads as High, Medium, or Low priority.
- Recommends a follow-up action.
- Exposes lead qualification through an MCP tool.
- Provides a REST API endpoint.
- Validates inputs and rejects invalid requests.

## Tech Stack

- Python
- FastMCP
- FastAPI
- Pydantic
- Uvicorn

## Project Files

- `lead_logic.py` — lead scoring and validation logic
- `mcp_server.py` — MCP server and tool
- `api.py` — FastAPI application
- `requirements.txt` — required packages

## Setup

Install dependencies:

```bash
pip install -r requirements.txt
```

## Run the API

From the project directory:

```bash
uvicorn api:app --reload
```

Open `http://127.0.0.1:8000/docs` in your browser
to test the API.

## API Endpoint

`POST /qualify-lead`

Example request:

```json
{
  "customer_name": "Aisha",
  "budget": 60000,
  "interest_level": "hot",
  "urgency": "high"
}
```

## Current Scope

Lead scoring uses predefined rules. The project currently
provides an MCP tool and a REST API; LLM-based autonomous
decision-making is not yet implemented.
"""

readme_file = project_dir / "README.md"
readme_file.write_text(readme_content.strip() + "\n", encoding="utf-8")

print("README created:", readme_file)
print("File exists:", readme_file.exists())


README created: C:\Users\ASUS\Documents\LeadPilot\README.md
File exists: True


In [25]:
for file in sorted(project_dir.iterdir()):
    print(file.name)

__pycache__
api.py
lead_logic.py
mcp_server.py
README.md
requirements.txt


In [26]:
%cd $project_dir

C:\Users\ASUS\Documents\LeadPilot


In [27]:
%run -m uvicorn api:app --host 127.0.0.1 --port 8000

RuntimeError: asyncio.run() cannot be called from a running event loop

C:\Users\ASUS\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3084: UserWarning: Unknown failure executing module: <uvicorn>
  warn('Unknown failure executing module: <%s>' % mod_name)


# LeadPilot — AI-Powered Lead Qualification MCP Server

A Python-based lead qualification project that scores potential customers based on their budget, interest level, and urgency.

## Project Overview

LeadPilot helps prioritize leads by assigning a score and recommending a follow-up action. It provides a lead-scoring logic module, an MCP tool, and a REST API built with FastAPI.

## Objectives

- Automate basic lead scoring.
- Classify leads into High, Medium, and Low priorities.
- Provide lead qualification through an MCP tool.
- Expose lead scoring through a REST API.

## Technologies Used

- Python
- FastMCP
- FastAPI
- Uvicorn
- Pydantic
- Jupyter Notebook